# FastAPI Assignment – Path Parameters, Query Parameters & POST
Complete answers and runnable Google Colab code for all 10 questions. Run cells in order.

## Setup

In [ ]:
!pip -q install fastapi uvicorn requests

## Q1. What is a path parameter?
A **path parameter** is a value included directly in a URL path to identify a resource. It is declared in curly braces in the route. In `/products/101`, `101` is the path parameter. If `/products/abc` is called when the parameter is typed as `int`, FastAPI returns **422 Unprocessable Entity** because `abc` cannot be converted to an integer.

In [ ]:
from fastapi import FastAPI
app = FastAPI()

@app.get("/products/{product_id}")
def get_product(product_id: int):
    return {"product_id": product_id, "message": "Product retrieved successfully"}

# /products/101 -> {"product_id": 101, "message": "Product retrieved successfully"}
# /products/abc -> HTTP 422 validation error

## Q2. What is a query parameter? How does it differ from a path parameter?
A **query parameter** is supplied after `?` in a URL, often to filter or customize results. A path parameter is part of the route path and generally identifies a specific resource; a query parameter is often optional when a default is provided. Example: `/products/101` has path parameter `101`; `/tasks?status=pending` has query parameter `status`.

In [ ]:
from fastapi import FastAPI
app = FastAPI()

@app.get("/tasks")
def get_tasks(status: str = "all", priority: str = "normal"):
    return {
        "status": status,
        "priority": priority,
        "message": f"Showing tasks with status '{status}' and priority '{priority}'"
    }

# /tasks -> defaults: status=all, priority=normal
# /tasks?status=pending&priority=high -> uses supplied values

## Q3. Identify path and query parameters in `/products/101?details=true`
`101` is the path parameter because it is part of the route path. `details=true` is the query parameter because it follows `?`.

Expected JSON:
- `/products/101` → `{"product_id":101,"details":false}`
- `/products/101?details=true` → `{"product_id":101,"details":true}`

In [ ]:
from fastapi import FastAPI
app = FastAPI()

@app.get("/products/{product_id}")
def get_product(product_id: int, details: bool = False):
    return {"product_id": product_id, "details": details}

## Q4. Identify and fix the mistakes
In the first endpoint, the route uses `{item_id}` but the function parameter is named `id`. The path parameter name must match the function parameter. Fix it by using `item_id` in the function.

In the second endpoint, `keyword: str` is required because it has no default. Calling `/search` without it causes HTTP 422. Give it a default such as `"all"` to make it optional.

In [ ]:
from fastapi import FastAPI
app = FastAPI()

@app.get("/items/{item_id}")
def get_item(item_id: int):
    return {"item_id": item_id}

@app.get("/search")
def search_items(keyword: str = "all"):
    return {"keyword": keyword}

# /items/10 -> {"item_id":10}
# /search -> {"keyword":"all"}

## Q5. Difference between GET and POST
**GET** retrieves data and should not change server state. **POST** submits data for processing or creation.

Examples where POST is suitable:
1. Submitting an online order: sends order details to create an order; GET is not suitable for this state-changing action.
2. Submitting a job application: sends applicant information for processing and storage; GET is intended for retrieval.

To test the endpoint in Swagger UI, open `/docs`, expand `POST /feedback`, click **Try it out**, then **Execute**.

In [ ]:
from fastapi import FastAPI
app = FastAPI()

@app.post("/feedback")
def submit_feedback():
    return {"message": "Feedback submitted successfully"}

# Swagger UI: http://127.0.0.1:8000/docs

## Q6. What is a request body? Define a Pydantic Book model.
A **request body** is data sent by a client in an HTTP request. It is commonly used with POST to submit structured data. A body is more suitable than putting all values in the URL when sending larger or structured objects. The Pydantic model validates the received fields.

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
app = FastAPI()

class Book(BaseModel):
    title: str
    genre: str
    available: bool

@app.post("/books")
def create_book(book: Book):
    return {"message": "Book received successfully", "book": book.model_dump()}

# Example JSON body:
# {"title":"The Alchemist","genre":"Fiction","available":true}

## Q7. Missing or incorrectly named Task fields
The model requires `title`, `status`, and `priority`. If `priority` is missing, FastAPI/Pydantic returns **HTTP 422** and identifies the missing field. If the client sends `task_title` instead of `title`, it does not satisfy the required `title` field. By default, Pydantic ignores extra fields, so the required `title` remains missing and validation fails. If the sample still omits `priority`, that field is missing too.

In [ ]:
from pydantic import BaseModel, ValidationError

class Task(BaseModel):
    title: str
    status: str
    priority: str

for data in [
    {"title": "Buy notebook", "status": "pending"},
    {"task_title": "Buy notebook", "status": "pending"}
]:
    try:
        print(Task(**data))
    except ValidationError as e:
        print(e)

## Q8. Common beginner mistakes and fixes
1. **Path parameter:** Route `{item_id}` but function uses `id`. Fix: use matching names.
2. **Query parameter:** Required parameter has no default although it should be optional. Fix: provide a default such as `keyword="all"`.
3. **POST request:** Using GET for an operation that submits or creates data. Fix: use POST for submission/creation.
4. **Request body:** Omitting required fields or misspelling field names. Fix: send every required field with the exact model field names and suitable types.

**Why restart Uvicorn in Colab?** A background Uvicorn process may still be running old imported code. Restarting stops the old process and loads the updated `main.py`, ensuring the latest routes and logic are served.

In [ ]:
# Restart Uvicorn after updating main.py in Colab
!pkill -f "uvicorn main:app" || true
!nohup uvicorn main:app --host 0.0.0.0 --port 8000 > uvicorn.log 2>&1 &

## Q9. Complete Library API
The app has a welcome endpoint, a book lookup endpoint with integer path parameter `book_id`, and a books endpoint with optional query parameter `genre` defaulting to `"all"`.

Expected:
- `/books/5` → `{"book_id":5,"message":"Book details requested successfully"}`
- `/books?genre=technology` → `{"genre":"technology","message":"Showing books in the 'technology' genre"}`


In [ ]:
from fastapi import FastAPI
app = FastAPI(title="Library API")

@app.get("/")
def home():
    return {"message": "Welcome to the Library API"}

@app.get("/books/{book_id}")
def get_book(book_id: int):
    return {"book_id": book_id, "message": "Book details requested successfully"}

@app.get("/books")
def get_books(genre: str = "all"):
    return {"genre": genre, "message": f"Showing books in the '{genre}' genre"}

## Q10. Complete Task Management API
The application below includes three sample tasks, a Pydantic model, lookup by task ID, filtering by optional status, and a POST endpoint that accepts a Task body and returns a confirmation. The POST endpoint returns the submitted task; it does not add it to the in-memory list because the question only asks for a confirmation with the received task.

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="Task Management API")

# (a) In-memory sample data
tasks = [
    {"id": 1, "title": "Complete assignment", "status": "pending", "priority": "high"},
    {"id": 2, "title": "Read Python notes", "status": "in_progress", "priority": "medium"},
    {"id": 3, "title": "Submit project", "status": "completed", "priority": "high"}
]

# (b) Pydantic model
class Task(BaseModel):
    title: str
    status: str
    priority: str

# (c) Find one task by ID
@app.get("/tasks/{task_id}")
def get_task(task_id: int):
    for task in tasks:
        if task["id"] == task_id:
            return task
    return {"message": "Task not found"}

# (d) Filter by status, or return all tasks
@app.get("/tasks")
def get_tasks(status: str = "all"):
    if status.lower() == "all":
        return tasks
    return [task for task in tasks if task["status"].lower() == status.lower()]

# (e) Accept Task in request body
@app.post("/tasks")
def create_task(task: Task):
    return {
        "message": "Task received successfully",
        "task": task.model_dump()
    }

# Run locally: uvicorn main:app --reload

### Q10 – Save the complete app and run it in Google Colab
The next cell writes the complete Q10 application to `main.py`, then starts Uvicorn in the background on port 8000.

In [ ]:
task_api_code = '''
from fastapi import FastAPI
from pydantic import BaseModel
app = FastAPI(title="Task Management API")
tasks = [
    {"id": 1, "title": "Complete assignment", "status": "pending", "priority": "high"},
    {"id": 2, "title": "Read Python notes", "status": "in_progress", "priority": "medium"},
    {"id": 3, "title": "Submit project", "status": "completed", "priority": "high"}
]
class Task(BaseModel):
    title: str
    status: str
    priority: str
@app.get("/tasks/{task_id}")
def get_task(task_id: int):
    for task in tasks:
        if task["id"] == task_id:
            return task
    return {"message": "Task not found"}
@app.get("/tasks")
def get_tasks(status: str = "all"):
    if status.lower() == "all":
        return tasks
    return [task for task in tasks if task["status"].lower() == status.lower()]
@app.post("/tasks")
def create_task(task: Task):
    return {"message": "Task received successfully", "task": task.model_dump()}
'''
with open("main.py", "w", encoding="utf-8") as f:
    f.write(task_api_code)
print("main.py created.")

In [ ]:
!pkill -f "uvicorn main:app" || true
!nohup uvicorn main:app --host 0.0.0.0 --port 8000 > uvicorn.log 2>&1 &

import time
time.sleep(3)
print("Uvicorn started on port 8000.")

### Test the endpoints using Requests
Use Requests to test the path parameter, query parameter, and POST request body.

In [ ]:
import requests

base = "http://127.0.0.1:8000"

# GET /tasks/{task_id}
r = requests.get(f"{base}/tasks/1")
print("GET /tasks/1:", r.status_code, r.json())

# Not-found case
r = requests.get(f"{base}/tasks/99")
print("GET /tasks/99:", r.status_code, r.json())

# GET /tasks (all tasks)
r = requests.get(f"{base}/tasks")
print("GET /tasks:", r.status_code, r.json())

# GET /tasks?status=pending
r = requests.get(f"{base}/tasks", params={"status": "pending"})
print("GET /tasks?status=pending:", r.status_code, r.json())

# POST /tasks with JSON request body
new_task = {"title": "Practice FastAPI", "status": "pending", "priority": "medium"}
r = requests.post(f"{base}/tasks", json=new_task)
print("POST /tasks:", r.status_code, r.json())

### Test using Swagger UI
For a local server, open `http://127.0.0.1:8000/docs`. In Colab, use the runtime's port-proxy feature for port 8000 if available; otherwise, the Requests cells above test the endpoints from inside Colab.

1. Expand `GET /tasks/{task_id}`, click **Try it out**, enter `1`, then **Execute**.
2. Expand `GET /tasks`, enter `pending` in `status` (or leave it blank to get all tasks), then **Execute**.
3. Expand `POST /tasks`, click **Try it out**, enter this JSON and click **Execute**:
   ```json
   {"title":"Practice FastAPI","status":"pending","priority":"medium"}
   ```

## End of Assignment
Run cells in order. The Q10 setup cell creates the complete `main.py` file.